# Per-video diagnostic - which videos are easiest / hardest

Evaluates every video separately (its own frames only) with Ultralytics' own metric,
for train, val and test. This shows
what the model finds easy vs hard and explains the val/test difference.

## 1. Setup

In [ ]:
!nvidia-smi
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/iadd_subset_v4
!unzip -q "/content/drive/MyDrive/iadd_subset_v4.zip" -d /content

## 2. Remove the leaked frames

In [ ]:
import os
import glob

DS = "/content/iadd_subset_v4"
exclude_file = "/content/drive/MyDrive/presentation/eval_exclude.txt"

with open(exclude_file) as f:
    for line in f:
        entry = line.strip()

        if entry == "":
            continue

        sp, stem = entry.split("/", 1)

        # Delete the listed image.
        image_path = f"{DS}/images/{sp}/{stem}.jpg"
        if os.path.exists(image_path):
            os.remove(image_path)

        # Delete its label.
        label_path = f"{DS}/labels/{sp}/{stem}.txt"
        if os.path.exists(label_path):
            os.remove(label_path)

# Delete label caches so they can be rebuilt.
cache_files = glob.glob(f"{DS}/labels/*.cache")
for cache_path in cache_files:
    os.remove(cache_path)

print("Excluded frames and labels removed")

## 3. Per-video evaluation

For each video, symlink only its frames into a temp folder and validate it. Records
images, instances, condition (D/N/R/A), mAP50 and mAP50-95. Saves a CSV to Drive.

In [ ]:
from ultralytics import YOLO
import os, glob, yaml, shutil
from collections import defaultdict
import pandas as pd

DS   = '/content/iadd_subset_v4'
RUNS = '/content/drive/MyDrive/presentation'
best = YOLO(f'{RUNS}/run5/weights/best.pt')
NAMES = ['person','car','motorcycle','bus','truck','traffic_light']

def per_video(split):
    groups = defaultdict(list)

    # Grouping frames based on videos
    for p in glob.glob(f'{DS}/images/{split}/*.jpg'):
        groups[os.path.basename(p).split('__')[0]].append(p)
    rows = []
    for k, (vid, imgs) in enumerate(sorted(groups.items()), 1):
        TMP = '/content/pv_tmp'

        # Preparing temp folder for videos
        shutil.rmtree(TMP, ignore_errors=True)
        os.makedirs(f'{TMP}/images/v')
        os.makedirs(f'{TMP}/labels/v')
        inst = 0
        for img in imgs:
            b = os.path.basename(img)
            os.symlink(img, f'{TMP}/images/v/{b}')  # Making links for images
            lb = f'{DS}/labels/{split}/' + b[:-4] + '.txt'
            if os.path.exists(lb):
                os.symlink(lb, f'{TMP}/labels/v/' + b[:-4] + '.txt')  # Making links for labels
                inst += sum(1 for _ in open(lb))  # counting instances in each video

        # Genrating a temp dataset and evaluating model on it
        yaml.safe_dump({'path': TMP, 'train': 'images/v', 'val': 'images/v', 'nc': 6, 'names': NAMES}, open(f'{TMP}/data.yaml', 'w'))
        m = best.val(data=f'{TMP}/data.yaml', split='val', verbose=False, plots=False, project='/content/pvruns', name='tmp', exist_ok=True)

        # Appending name of videos, conditions, number of images and instances, mAP50, and mAP50-95
        rows.append({'split': split, 'video': vid, 'cond': vid.split('_')[-1],
                     'images': len(imgs), 'instances': inst,
                     'mAP50': round(float(m.box.map50), 4), 'mAP50_95': round(float(m.box.map), 4)})

        # progress after evaluating 10 videos
        if k % 10 == 0:
            print(f'  {split}: {k}/{len(groups)}', flush=True)
    return rows

all_rows = []
for split in ('val', 'test', 'train'):
    print(f'=== evaluating {split} videos ===', flush=True)
    all_rows += per_video(split)

df = pd.DataFrame(all_rows)
df.to_csv(f'{RUNS}/pervideo_run5.csv', index=False)
print('\nsaved -> pervideo_run5.csv')